In [1]:
_cf_value_cache = {}

def cf_value(terms):
    # return continued_fraction of terms as a real number (cached)
    key = tuple(tuple(part) for part in terms)
    if key not in _cf_value_cache:
        _cf_value_cache[key] = continued_fraction(terms).n()
    return _cf_value_cache[key]

def max_value_of_lambda (negative_side, positive_side, a_0 = 0):
    # return maximum value of lambda_0(A). A = (a_{-h}, ..., a_{-1} |a_0 (= 0),| a_1, ..., a_k)
    # negative_side = [a_{-1}, ..., a_{-h}]
    # positive_side = [a_1, ..., a_k]

    # if n is even, to get maximum [0;a_1, ..., a_n, ...], we need to extend by (1, 3). if k is odd, by (3, 1).\
    if len(positive_side) % 2 == 0:
        positive_extension = (1,3)
    else:
        positive_extension = (3,1)

    if len(negative_side) % 2 == 0:
        negative_extension = (1,3)
    else:
        negative_extension = (3,1)

    return (
        RR(a_0) + cf_value([(0, *positive_side), positive_extension]) + cf_value([(0, *negative_side), negative_extension]),
        (a_0, negative_side, negative_extension, positive_side, positive_extension)
    )

def max_value_of_lambda_i (negative_side, positive_side, a_0):
    # return maximum value of lambda_i(A). A = (a_{-h}, ..., a_{-1} |a_0 = 0, a_1, ..., a_k), i ≠ 0
    # negative_side = [a_{-1}, ..., a_{-h}]
    # positive_side = [a_1, ..., a_k]

    sequence_around_zero = negative_side[::-1] + [a_0] + positive_side
    position_zero = len(negative_side)

    max_values = []
    for i in range(len(sequence_around_zero)):
        if i == position_zero:
            continue  # skip the position corresponding to a_0
        neg = sequence_around_zero[:i][::-1]  # elements before i, reversed
        a_0 = sequence_around_zero[i]          # i-th element as a_0
        pos = sequence_around_zero[i+1:]      # elements after i
        val, expression = max_value_of_lambda(neg, pos, a_0)
        max_values.append((val, expression))
    max_values.append((RR(sqrt(21)), (3, [], (1, 3), [], (1, 3))))

    if not max_values:
        return None, None

    max_value, max_expression = max(max_values, key=lambda item: RR(item[0]))
    return max_value, max_expression

def _cf_candidates(side, family):
    extensions = ((1, 3), (3, 1)) if family == "eta" else ((1, 2), (2, 1))
    return [(cf_value([(0, *side), ext]), ext) for ext in extensions]

def _pick_cf(side, family, which):
    pairs = _cf_candidates(side, family)
    return (min if which == "min" else max)(pairs, key=lambda item: item[0])

def _t_interval_expression(negative_side, neg_ext, positive_side, pos_ext):
    return (0, negative_side, neg_ext, positive_side, pos_ext)

def T_interval (negative_side, positive_side):
    # return beginning/end of T_interval I(negative_side | positive_side),
    # together with expressions witnessing each endpoint

    beginning_candidates = []
    end_candidates = []

    for pos_family, neg_family in [("eta", "theta"), ("theta", "eta")]:
        pos_b, pos_ext_b = _pick_cf(positive_side, pos_family, "min")
        neg_b, neg_ext_b = _pick_cf(negative_side, neg_family, "min")
        beginning_candidates.append((
            pos_b + neg_b,
            _t_interval_expression(negative_side, neg_ext_b, positive_side, pos_ext_b),
        ))

        pos_e, pos_ext_e = _pick_cf(positive_side, pos_family, "max")
        neg_e, neg_ext_e = _pick_cf(negative_side, neg_family, "max")
        end_candidates.append((
            pos_e + neg_e,
            _t_interval_expression(negative_side, neg_ext_e, positive_side, pos_ext_e),
        ))

    beginning, beginning_expr = min(beginning_candidates, key=lambda item: item[0])
    end, end_expr = max(end_candidates, key=lambda item: item[0])
    return beginning, beginning_expr, end, end_expr

def T_ratio (negative_side, positive_side):
    # return T_ratio of I(negative_side | positive_side)
    positive_side_eta = [cf_value([(0, *positive_side), (1,3)]), cf_value([(0, *positive_side), (3,1)])]
    negative_side_eta = [cf_value([(0, *negative_side), (1,3)]), cf_value([(0, *negative_side), (3,1)])]

    return abs( (positive_side_eta[0] - positive_side_eta[1]) / (negative_side_eta[0] - negative_side_eta[1]) )

def is_addmissible_interval(negative_side, positive_side):
    # return True if I(negative_side | positive_side) is addmissible (zulässig), False otherwise
    v = T_ratio(negative_side, positive_side)

    return (
        (len(positive_side) >= 2 or positive_side[0] >= 2)
        and (len(negative_side) >= 2 or negative_side[0] >= 2)
        and (5/17 <= v and v <= 17/5)
    )

In [2]:
from itertools import product
import time

FIRST_NONNEGATIVE_DIGITS = (4, 5)
EXTENSION_DIGITS = (1, 2, 3, 4)

def with_a_0(expression, a_0):
    return (a_0, expression[1], expression[2], expression[3], expression[4])

def process_interval(negative_side, nonnegative_side, algebraic=False):
    # only keep admissible pairs
    if not is_addmissible_interval(negative_side, nonnegative_side[1:]):
        return None
    beginning, beginning_expr, end, end_expr = T_interval(
        negative_side,
        nonnegative_side[1:],
    )
    maximum_lambda_i, lambda_i_expr = max_value_of_lambda_i(
        negative_side,
        nonnegative_side[1:],
        nonnegative_side[0],
    )
    a_0 = nonnegative_side[0]

    if maximum_lambda_i > end + a_0:
        return None

    interval_start = max(beginning + a_0, maximum_lambda_i)
    interval_end = end + a_0
    if RR(maximum_lambda_i) >= RR(beginning + a_0):
        lower_expr = lambda_i_expr
    else:
        lower_expr = with_a_0(beginning_expr, a_0)

    return {
        "negative_side": negative_side,
        "nonnegative_side": nonnegative_side,
        "interval": [interval_start, interval_end],
        "lower_expr": lower_expr,
        "upper_expr": with_a_0(end_expr, a_0),
    }

def sequences_of_length(length, digits=EXTENSION_DIGITS):
    if length == 0:
        yield []
    else:
        for seq in product(digits, repeat=length):
            yield list(seq)

def nonnegative_sides_of_length(length):
    # length >= 1; the first digit is from 1..5, later digits from 1..4
    if length == 1:
        for d in FIRST_NONNEGATIVE_DIGITS:
            yield [d]
    else:
        for first in FIRST_NONNEGATIVE_DIGITS:
            for tail in product(EXTENSION_DIGITS, repeat=length - 1):
                yield [first] + list(tail)

def for_all_intervals(max_depth):
    # depth = len(negative_side) + len(nonnegative_side)
    # start from nonnegative_side of length 1; enumerate pairs with depth <= max_depth
    for total_depth in range(1, max_depth + 1):
        for neg_len in range(total_depth + 1):
            nonneg_len = total_depth - neg_len
            if nonneg_len < 1:
                continue
            for negative_side in sequences_of_length(neg_len):
                for nonnegative_side in nonnegative_sides_of_length(nonneg_len):
                    yield negative_side, nonnegative_side

def count_interval_pairs(max_depth):
    # depth d contributes d * 5 * 4^(d-1)
    return sum(d * len(FIRST_NONNEGATIVE_DIGITS) * len(EXTENSION_DIGITS) ** (d - 1) for d in range(1, max_depth + 1))

def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    if hours > 0:
        return f"{hours:d}:{minutes:02d}:{seconds:02d}"
    return f"{minutes:02d}:{seconds:02d}"

def print_progress(current, total, start_time, found_count, bar_width=30):
    current = int(current)
    total = int(total)
    found_count = int(found_count)
    elapsed = float(time.time() - start_time)
    ratio = current / total if total else 1.0
    filled = min(bar_width, int(bar_width * ratio))
    bar = "#" * filled + "-" * (bar_width - filled)
    rate = current / elapsed if elapsed > 0 else 0.0
    remaining = (total - current) / rate if rate > 0 else 0.0
    print(
        f"\r[{bar}] {100 * ratio:6.2f}% "
        f"({current:,}/{total:,}) elapsed={format_seconds(elapsed)} "
        f"ETR={format_seconds(remaining)} found={found_count}",
        end="",
        flush=True,
    )

def search_intervals(max_depth, process=process_interval, progress_step=1000, **process_kwargs):
    # apply process to every pair with len(negative_side) + len(nonnegative_side) <= max_depth
    total = count_interval_pairs(max_depth)
    results = []
    start_time = time.time()

    for index, (negative_side, nonnegative_side) in enumerate(for_all_intervals(max_depth), start=1):
        try:
            record = process(negative_side, nonnegative_side, **process_kwargs)
        except (IndexError, ZeroDivisionError, ValueError, ArithmeticError):
            record = None
        if record is not None:
            results.append(record)

        if index == 1 or index % progress_step == 0 or index == total:
            print_progress(index, total, start_time, len(results))

    print()
    return results

In [3]:
def format_bilateral_cf(witness, label):
    # A = (a_{-h}, ..., a_{-1} | a_0, a_1, ..., a_k)
    negative_side = witness["negative_side"]
    nonnegative_side = witness["nonnegative_side"]
    neg = list(reversed(negative_side))
    pos = list(nonnegative_side[1:])
    neg_str = ", ".join(str(a) for a in neg)
    pos_str = ", ".join(str(a) for a in pos)
    neighbor_str = f"({neg_str} | {pos_str})"
    expression = witness["lower_expr"] if label == "lower" else witness["upper_expr"]

    return nonnegative_side[0], neighbor_str, expression

def cf_record_sort_key(record):
    return (tuple(record["negative_side"]), tuple(record["nonnegative_side"]))

def select_endpoint_witnesses(records, interval_start, interval_end):
    interval_start = RR(interval_start)
    interval_end = RR(interval_end)

    lower_candidates = [
        record for record in records
        if RR(record["interval"][0]) == interval_start
    ]
    upper_candidates = [
        record for record in records
        if RR(record["interval"][1]) == interval_end
    ]

    if not lower_candidates:
        min_start = min(RR(record["interval"][0]) for record in records)
        lower_candidates = [
            record for record in records
            if RR(record["interval"][0]) == min_start
        ]
    if not upper_candidates:
        max_end = max(RR(record["interval"][1]) for record in records)
        upper_candidates = [
            record for record in records
            if RR(record["interval"][1]) == max_end
        ]

    return (
        min(lower_candidates, key=cf_record_sort_key),
        min(upper_candidates, key=cf_record_sort_key),
    )

def merge_interval_records(records):
    # merge overlapping or touching intervals, keeping lower/upper CF witnesses
    valid_records = [
        record
        for record in records
        if RR(record["interval"][0]) <= RR(record["interval"][1])
    ]
    if not valid_records:
        return []

    valid_records.sort(
        key=lambda record: (
            RR(record["interval"][0]),
            RR(record["interval"][1]),
        )
    )

    first = valid_records[0]
    merged = [{
        "interval": [RR(first["interval"][0]), RR(first["interval"][1])],
        "records": [first],
    }]
    for record in valid_records[1:]:
        start = RR(record["interval"][0])
        end = RR(record["interval"][1])
        if start <= merged[-1]["interval"][1]:
            merged[-1]["interval"][1] = max(merged[-1]["interval"][1], end)
            merged[-1]["records"].append(record)
        else:
            merged.append({
                "interval": [start, end],
                "records": [record],
            })

    for component in merged:
        lower, upper = select_endpoint_witnesses(
            component["records"],
            component["interval"][0],
            component["interval"][1],
        )
        component["lower_witness"] = lower
        component["upper_witness"] = upper
        del component["records"]

    return merged

def disjoint_union_of_intervals(records):
    return merge_interval_records(records)

def show_merged_intervals(filled_intervals):
    merged_union = disjoint_union_of_intervals(filled_intervals)
    print("merged component count:", len(merged_union))
    for index, component in enumerate(merged_union, start=1):
        start, end = component["interval"]
        print(f"component {index}: [{n(start)}, {n(end)}]")
        for label, witness in [
            ("lower", component["lower_witness"]),
            ("upper", component["upper_witness"]),
        ]:
            a_0, neighbor_interval, cf_expression = format_bilateral_cf(witness, label)
            print(
                f"  {label}|",
                "a_0:",
                a_0,
                "T-interval:",
                neighbor_interval,
                "expression:",
                cf_expression,
            )
        print()


for max_depth in range(7, 13):
    pair_count = count_interval_pairs(max_depth)
    print("| max_depth:", max_depth, "| pair count:", f"{pair_count:,}", "|")
    filled_intervals = search_intervals(max_depth)
    show_merged_intervals(filled_intervals)
    print("-" * 100)

| max_depth: 7 | pair count: 72,818 |
[##############################] 100.00% (72,818/72,818) elapsed=00:01 ETR=00:00 found=4498
merged component count: 5
component 1: [4.58257569495584, 4.60555422895806]
  lower| a_0: 4 T-interval: (3 | 3) expression: (3, [], (1, 3), [], (1, 3))
  upper| a_0: 4 T-interval: (2, 1, 2 | 4, 3) expression: (4, [2, 1, 2], (3, 1), [4, 3], (1, 2))

component 2: [4.60571502519474, 4.61757718177764]
  lower| a_0: 4 T-interval: (2, 3, 3 | 3, 3, 3) expression: (4, [3, 3, 2], (1, 3), [3, 3, 3], (1, 2))
  upper| a_0: 4 T-interval: (1, 1, 1, 2 | 4, 3) expression: (4, [2, 1, 1, 1], (1, 3), [4, 3], (1, 2))

component 3: [4.61834896178803, 4.61872299816472]
  lower| a_0: 4 T-interval: (3, 4, 3 | 3, 4, 3) expression: (4, [3, 4, 3], (1, 2), [3, 4, 3], (1, 3))
  upper| a_0: 4 T-interval: (3, 4, 3 | 3, 4, 3) expression: (4, [3, 4, 3], (2, 1), [3, 4, 3], (3, 1))

component 4: [4.61872673838388, 6.61772140424125]
  lower| a_0: 4 T-interval: (2, 1, 1, 2 | 4, 3) expression: (